# AI Agent MCP Demo

Minimal LangChain proof of concept that loads tools from the asyncroscopy MCP server and asks the model to use one or two of them. This will be useful to test our MCP server and new tools.

## Prereqs

Start the Tango stack and MCP server first:

```bash
uv run startup_scripts/run_servers.py
uv run startup_scripts/run_mcp.py --yaml configs/mcp.yaml
```

Install the optional AI extras:

```bash
uv sync --extra agent
# for local models do uv sync --extra agent --extra localagent
```

In [1]:
import os
from pprint import pprint

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_core.tools import BaseTool

### Connect to the running MCP server

In [2]:
MCP_URL = "http://127.0.0.1:8000/mcp"

client = MultiServerMCPClient(
    {
        "asyncroscopy": {
            "url": MCP_URL,
            "transport": "streamable_http",
        }
    }
)

tools = await client.get_tools()
print(f"Loaded {len(tools)} tool(s)")
print([tool.name for tool in tools])

Loaded 45 tool(s)
['get_data_from_key', 'list_devices', 'CAMERA_State', 'CAMERA_Status', 'DATA_State', 'DATA_Status', 'DATA_configure', 'DATA_get_config', 'DATA_register_path', 'DATA_register_save_path', 'DATA_start_tiled_server', 'DATA_stop_tiled_server', 'EDS_State', 'EDS_Status', 'FLUCAM_State', 'FLUCAM_Status', 'DigitalTwin_Connect', 'DigitalTwin_Disconnect', 'DigitalTwin_State', 'DigitalTwin_Status', 'DigitalTwin_acquire_camera_image', 'DigitalTwin_acquire_flucam_image', 'DigitalTwin_acquire_scanned_data_advanced', 'DigitalTwin_acquire_scanned_image', 'DigitalTwin_acquire_spectrum', 'DigitalTwin_auto_focus', 'DigitalTwin_blank_beam', 'DigitalTwin_get_defocus', 'DigitalTwin_get_fov', 'DigitalTwin_get_image_data_cached', 'DigitalTwin_get_screen_current', 'DigitalTwin_get_stage', 'DigitalTwin_move_stage', 'DigitalTwin_place_beam', 'DigitalTwin_place_beam_list', 'DigitalTwin_set_column_valves', 'DigitalTwin_set_defocus', 'DigitalTwin_set_fov', 'DigitalTwin_set_image_shift', 'DigitalTw

In [8]:
def filter_tools(tools: list[BaseTool], tool_names: list[str]) -> list[BaseTool]:
    return [tool for tool in tools if tool.name in tool_names]

# Filter the tools so we can test a few at a time
tools = filter_tools(tools, ["list_devices", "DigitalTwin_acquire_scanned_image"])
print(f"Filtered to {len(tools)} tool(s): {[tool.name for tool in tools]}")

Filtered to 2 tool(s): ['list_devices', 'DigitalTwin_acquire_scanned_image']


### Run cell below to use an OPENAI-COMPATIBLE model

In [ ]:
model = init_chat_model(
    model="YOUR MODEL NAME",
    model_provider="openai, google_genai, etc.", # Make sure to install the right provider package for the model you want to use
    api_key="YOUR API KEY", 
)

using_local_model = False

### Run cell below to use a LOCAL model

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

HF_MODEL_ID = r"C:\Users\Public\Desktop\Agents\qwen-14b"

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
hf_model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

hf_pipeline = pipeline(
    "text-generation",
    model=hf_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.2,
    return_full_text=False,
)

model = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=hf_pipeline))

using_local_model = True

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


### Basic prompt

In [ ]:
agent = create_agent(model, tools)

def final_text(result):
    message = result["messages"][-1]
    return getattr(message, "content", message)

if not using_local_model:
    result = await agent.ainvoke({
        "messages": "List the available devices with the tool in one short sentence, then stop."
    })
else :
    result = agent.invoke(
        {"messages": "List the available devices with the tool in one short sentence, then stop."},
        config={"configurable": {"use_async": False}}
    )

print(final_text(result))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


I'm sorry for any misunderstanding, but as an AI model, I don't have direct access to invoke system tools or commands on external devices or systems. I can help you understand how to use the `list_devices` command if you're looking for information on that. Could you please provide more context or clarify your request?


### Test image acquisition

In [7]:
if not using_local_model:
    result = await agent.ainvoke({
        "messages": "Get a scanned image."
    })
else :
    result = agent.invoke(
        {"messages": "Get a scanned image."},
        config={"configurable": {"use_async": False}}
    )

print(final_text(result))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I'm sorry for the misunderstanding, but as a text-based AI, I don't have the capability to get or display images directly. However, if you need help with scanned images—such as how to scan them, edit them, or convert them into different formats—I can certainly provide guidance on those topics. Could you please specify what you would like to do with a scanned image?
